In [2]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Operator, Statevector
from qiskit_aer import AerSimulator
 
def fourier_negative(n):
    qc = QuantumCircuit(n, name="F_neg")
    for target in reversed(range(n)):
        qc.h(target)
        for control in reversed(range(target)):
            qc.cp(-np.pi / 2**(target - control), control, target)
    for q in range(n // 2):
        qc.swap(q, n - q - 1)
    return qc
 
def qpe_z(n, eigen=1, measured=True):
    qc = QuantumCircuit(n + 1, n + 1)
    if eigen == 1:
        qc.x(0)  # b register
    for q in range(1, n + 1):
        qc.h(q)
    qc.cz(1, 0)  # Z^1; controlled-Z^2 and higher are identity
    qc.barrier()
    qc.compose(fourier_negative(n), range(1, n + 1), inplace=True)
    if measured:
        for q in range(1, n + 1):
            qc.measure(q, q)  # classical bit 0 stays zero
    return qc
sim = AerSimulator(seed_simulator=2569)
for n in (2, 3):
    for eigen in (0, 1):
        circuit = qpe_z(n, eigen)
        counts = sim.run(
            transpile(circuit, sim, seed_transpiler=2569),
            shots=1024
        ).result().get_counts()
        print(f"n={n}, eigen={eigen}:", counts)
 
sv = Statevector.from_instruction(qpe_z(3, 1, measured=False))
print("Full quantum state:", format(np.argmax(sv.probabilities()), "04b"))
full_measurement = qpe_z(3, 1)
full_measurement.measure(0, 0)
full_counts = sim.run(
    transpile(full_measurement, sim), shots=1024
).result().get_counts()
print("All four qubits measured:", full_counts)


n=2, eigen=0: {'000': 1024}
n=2, eigen=1: {'100': 1024}
n=3, eigen=0: {'0000': 1024}
n=3, eigen=1: {'1000': 1024}
Full quantum state: 1001
All four qubits measured: {'1001': 1024}
